# 09.5 音乐推荐：从收听历史到候选排序

前面四节都有用户主动提交的显式查询。推荐系统不一定接收这种查询，而是根据用户历史、当前会话、时间或场景等条件对候选内容排序。本 Notebook 只使用收听历史，问题是：该用户尚未听过的歌曲中，哪些应排在前面？

从检索角度看，用户及其状态构成查询条件，曲库仍是候选集合；但推荐还涉及探索、新颖性、多样性和业务约束，并不完全等同于相关性检索。

纯协同过滤（collaborative filtering）只使用用户与歌曲的交互。user-based 方法参考相似用户的行为；item-based 方法根据歌曲的共现关系打分。本 Notebook 实现后一种。

评分是显式反馈，播放次数是隐式反馈。未观察到播放通常不代表负面偏好，重复播放也可能受自动播放、共享账号或场景影响。算法可以从未观察交互中抽取负例或赋予较低置信度，即使用户未明确给出负反馈。

## 0. 一个小型共听矩阵

示例含十位用户和二十首歌。歌曲分为民歌、流行、摇滚、古典四个风格簇，每簇五首；每位用户收听五到八首，播放次数为一到八。
主听簇和少量跨簇收听被预先写入矩阵，以便观察共现相似度如何反映这些结构。


In [ ]:
from pathlib import Path
import sys

# 路径推断：从 cwd 向上找含 CODE/chapter09/_common 的目录；ROOT 指向 CODE/chapter09/
_p = Path.cwd()
while not (_p / "CODE" / "chapter09" / "_common").exists():
    _parent = _p.parent
    if _parent == _p:
        raise FileNotFoundError("未找到项目根目录（包含 CODE/chapter09/_common 的目录），请在项目内运行本 Notebook")
    _p = _parent
ROOT = _p / "CODE" / "chapter09"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from _common.env_check import check_notebook_env
from _common.paths import portable_path
from _common.plotting import GRAY_IMAGE_CMAP, finish_figure, setup_plot_style

check_notebook_env("09_5_recommendation_toy")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

OUTPUT_FIGURES = ROOT / "output_figures"
OUTPUT_TABLES = ROOT / "outputs" / "tables"
for path in [OUTPUT_FIGURES, OUTPUT_TABLES]:
    path.mkdir(parents=True, exist_ok=True)
setup_plot_style()

def rel(path):
    return portable_path(path, ROOT)

clusters = {"民": ["民歌"] * 5, "流": ["流行"] * 5, "摇": ["摇滚"] * 5, "古": ["古典"] * 5}
songs = [f"{c}{i}" for c in clusters for i in range(5)]
genre_of = {f"{c}{i}": clusters[c][i] for c in clusters for i in range(5)}
users = [f"U{k}" for k in range(10)]

# 每位用户的主听簇与播放次数，手工排出簇结构；民4 是冷门歌，只有 U2 听过一次
listens = {
    "U0": {"民0": 5, "民1": 3, "民2": 7, "流0": 4, "流1": 6, "流2": 2},
    "U1": {"民0": 8, "民1": 6, "民3": 4, "民2": 2, "流0": 1},
    "U2": {"民1": 3, "民4": 1, "古0": 5, "古1": 4, "古2": 2},
    "U3": {"流0": 7, "流1": 5, "流3": 3, "流4": 6, "流2": 2, "民0": 1},
    "U4": {"流1": 6, "流4": 4, "摇0": 5, "摇1": 7, "摇2": 3, "流0": 2, "摇3": 1, "流2": 1},
    "U5": {"摇0": 8, "摇1": 4, "摇3": 6, "摇4": 2, "摇2": 1},
    "U6": {"摇2": 7, "摇4": 5, "古1": 3, "古3": 6, "古4": 4, "摇0": 1},
    "U7": {"古0": 6, "古2": 8, "古3": 2, "古4": 5, "古1": 1},
    "U8": {"古1": 7, "古4": 3, "民0": 2, "民3": 6, "民1": 1},
    "U9": {"流2": 8, "流3": 4, "民2": 5, "民3": 3, "民0": 1, "流0": 2, "民1": 1},
}
M = pd.DataFrame(0, index=users, columns=songs)
for u, items in listens.items():
    for s, n in items.items():
        M.loc[u, s] = n
print(f"共听矩阵 {M.shape[0]} 用户 × {M.shape[1]} 首,非零 {int((M > 0).sum().sum())} 格,密度 {(M > 0).mean().mean():.2f}")

fig, ax = plt.subplots(figsize=(8.5, 3.6))
ax.imshow(M.to_numpy(), aspect="auto", cmap=GRAY_IMAGE_CMAP)
ax.set_xticks(range(len(songs)), songs, fontsize=8)
ax.set_yticks(range(len(users)), users, fontsize=8)
for k in (5, 10, 15):
    ax.axvline(k - 0.5, color="0.1", linewidth=0.8)
ax.set_xlabel("歌曲（按风格簇排列）")
finish_figure(fig, OUTPUT_FIGURES / "09_5_user_item_matrix.png")
plt.show()


## 1. item-based：被同一批人听的歌彼此相似

item-based 协同过滤把每首歌表示为十位用户的播放次数向量，并用余弦相似度比较方向。听众重合会提高点积，但结果还取决于各用户播放次数的相对分布。活跃用户和高播放次数可能产生较大影响。

为某位用户排序时，代码按其播放次数加权累加候选歌曲与已听歌曲的相似度，再排除已听歌曲。这个分数是本 Notebook 定义的启发式排序量，不是点击、播放或喜欢的概率。
相似度只来自当前行为矩阵，不含音频或乐理信息。U5 已听过全部五首摇滚；排除已听歌曲后，前三名是流1、流4和古3。原因是 U4 同时收听流行与摇滚，U6 同时收听摇滚与古典，使相关歌曲的听众向量出现重合。该例说明推荐结果会继承输入矩阵中的共现结构；这种行为相似只反映当前交互，不代表歌曲内容本身相似。


In [ ]:
# 歌曲 × 歌曲余弦相似度
X = M.to_numpy(dtype=float).T  # 20 首 × 10 用户
norms = np.linalg.norm(X, axis=1, keepdims=True)
sim = (X @ X.T) / (norms @ norms.T + 1e-12)
sim_df = pd.DataFrame(sim, index=songs, columns=songs)

fig, ax = plt.subplots(figsize=(5.8, 4.8))
img = ax.imshow(sim, cmap=GRAY_IMAGE_CMAP, vmin=0, vmax=1)
ax.set_xticks(range(len(songs)), songs, fontsize=7, rotation=90)
ax.set_yticks(range(len(songs)), songs, fontsize=7)
for k in (5, 10, 15):
    ax.axvline(k - 0.5, color="0.1", linewidth=0.8)
    ax.axhline(k - 0.5, color="0.1", linewidth=0.8)
fig.colorbar(img, ax=ax, fraction=0.046, pad=0.04)
finish_figure(fig, OUTPUT_FIGURES / "09_5_item_similarity.png")
plt.show()


def recommend(user, top=3):
    # 听过的歌按播放次数加权，汇总相似歌得分，排除已听
    heard = M.loc[user]
    scores = (sim_df * heard).sum(axis=1)
    scores[heard > 0] = -np.inf
    return scores.sort_values(ascending=False).head(top)


for user in ["U0", "U5"]:
    top3 = recommend(user)
    heard_genres = "/".join(sorted({genre_of[s] for s in M.columns[M.loc[user] > 0]}))
    print(f"{user}(听过：{heard_genres})的 Top-3 推荐：")
    print(pd.DataFrame({"歌曲": top3.index, "风格": [genre_of[s] for s in top3.index], "得分": top3.round(2).to_numpy()}).to_string(index=False))


## 2. 冷启动：交互不足时的局限

只依赖交互的协同过滤难以处理缺少行为的新用户或新物品，这类问题统称冷启动（cold start）。冷门物品虽有非零向量，但相似度可能由极少数用户决定，对单条交互较敏感。
在本实现中，新歌对应零向量，余弦在数学上未定义。代码在分母加入极小量，因此把它与所有歌曲的相似度算为零。除非另有默认排序、随机探索或并列分数处理，它不会凭当前相似度得到优先推荐。
冷门歌民4只有 U2 听过一次。删除这条交互后，它在当前计算中与所有歌曲的相似度都变为零。这个受控例子展示的是单条行为对稀疏向量的影响，不表示所有冷门歌都会出现同样结果。

引入音频、文本、艺人或目录元数据，可以为新歌构造内容特征。矩阵分解则利用共享低维因素和正则化，从有限交互中估计用户与歌曲向量；交互很少时估计仍可能不稳定，全无交互的新歌也无法仅靠协同信号学得可靠向量。
行为嵌入与内容嵌入来源不同。混合推荐可以联合使用二者，但具体组合方式及其收益需要另行训练和评估。


In [ ]:
# 冷门歌民4：只有 U2 听过一次。删除这一次收听，它的相似度立刻全零
print("删除前民4 与民歌簇的相似度:")
print(sim_df.loc["民4", ["民0", "民1", "民2", "民3"]].round(3).to_string())

M_cold = M.copy()
M_cold.loc["U2", "民4"] = 0
Xc = M_cold.to_numpy(dtype=float).T
norms_c = np.linalg.norm(Xc, axis=1, keepdims=True)
sim_cold = (Xc @ Xc.T) / (norms_c @ norms_c.T + 1e-12)
sim_cold_df = pd.DataFrame(sim_cold, index=songs, columns=songs)
print()
print("删除 U2 的收听之后民4 与民歌簇的相似度:")
print(sim_cold_df.loc["民4", ["民0", "民1", "民2", "民3"]].round(3).to_string())

# 新歌入库：一行全零，与任何歌的相似度都是零
M_new = M.copy()
M_new["民5"] = 0
Xn = M_new.to_numpy(dtype=float).T
norms_n = np.linalg.norm(Xn, axis=1, keepdims=True)
sim_new = (Xn @ Xn.T) / (norms_n @ norms_n.T + 1e-12)
sim_new_df = pd.DataFrame(sim_new, index=M_new.columns, columns=M_new.columns)
print()
print("新歌民5(无人听过)与民歌簇的相似度:")
print(sim_new_df.loc["民5", ["民0", "民1", "民2", "民3"]].round(3).to_string())


## 3. 真实数据与适用范围

Million Song Dataset 本体提供一百万首歌曲的预抽音频特征与元数据，不含音频文件，也不含用户收听记录。与之配套的 Taste Profile Subset 提供用户行为：48,373,586 条（用户、歌曲、播放次数）三元组，覆盖 1,019,318 位用户和 384,546 首歌曲。
Last.fm 配套数据另含歌曲级标签与歌曲间相似度，三者的数据角色不同。Taste Profile 官方页面还提示，部分歌曲与 MSD 曲目的映射曾出现错误，并提供了修正信息。使用时应保留这一数据质量说明。
2012 年 Million Song Dataset Challenge 使用 Taste Profile 的部分收听历史，让系统预测被留出的歌曲。这是集合补全式的离线评估。
实际推荐系统可能组合行为、内容、上下文、流行度、编辑规则和探索策略。


In [ ]:
sim_df.to_csv(OUTPUT_TABLES / "09_5_item_similarity.csv")
rec_rows = []
for user in users:
    for rank, (song, score) in enumerate(recommend(user).items(), 1):
        rec_rows.append({"user": user, "rank": rank, "song": song, "genre": genre_of[song], "score": round(float(score), 3)})
pd.DataFrame(rec_rows).to_csv(OUTPUT_TABLES / "09_5_recommendations.csv", index=False)
print("表格已写入", rel(OUTPUT_TABLES))
print("图已写入", rel(OUTPUT_FIGURES))
